# Compute demo
Runs for about 2 minutes in three phases, so the run page's charts and the **Compute** page have something to show:
1. **Memory:** grows by 16 MiB a second, to about 480 MiB.
2. **CPU:** keeps one core busy.
3. **Disk:** writes files into the space (`/space`), not `/tmp`, which counts as memory.

Start it in several spaces at once to see runs spread over the workers, and queue when they're all busy.

In [ ]:
import os, platform, time
print('running on', platform.node(), flush=True)  # the run's container (or Pod)

In [ ]:
# 1. Memory: 30 steps of 16 MiB, one a second
blocks = []
for i in range(30):
    blocks.append(bytearray(16 << 20))
    print(f'memory step {i + 1}/30: {len(blocks) * 16} MiB', flush=True)
    time.sleep(1)

In [ ]:
# 2. CPU: 40 seconds of arithmetic on one core
t0, n = time.time(), 0
while time.time() - t0 < 40:
    n += sum(i * i for i in range(100_000))
    if int(time.time() - t0) % 10 == 0:
        print(f'cpu: {int(time.time() - t0)} s', flush=True)
print('cpu done', flush=True)

In [ ]:
# 3. Disk: 20 files of 8 MiB in the space, then removed
os.makedirs('demo-out', exist_ok=True)
for i in range(20):
    with open(f'demo-out/part-{i}.bin', 'wb') as f:
        f.write(os.urandom(8 << 20))
        f.flush(); os.fsync(f.fileno())
    print(f'disk: {(i + 1) * 8} MiB written', flush=True)
    time.sleep(1)
for name in os.listdir('demo-out'):
    os.remove(os.path.join('demo-out', name))
os.rmdir('demo-out')

In [ ]:
del blocks
print('done', flush=True)